# Проверка генерации на одной T4 в Kaggle
Выберите GPU T4 x2 и включите Internet. Добавьте приватный Dataset с checkpoint_130m.zip и questions_used.jsonl. Выполните Run All. Код использует только GPU 0. Если Kaggle распаковал архив в Dataset, ноутбук соберёт его обратно до начала замера.


In [ ]:
%pip install -q "transformers==4.48.3" "tokenizers==0.21.0" "huggingface-hub==0.36.2" "safetensors>=0.4.5"


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
assert INPUT.is_dir() and WORK.is_dir(), 'Запустите этот ноутбук в Kaggle'
os.chdir(WORK)

def find_unique(name):
    matches = sorted(INPUT.rglob(name))
    assert len(matches) == 1, f'Ожидался один {name}, найдено {len(matches)}. Проверьте Add Input.'
    return matches[0]

shutil.copy2(find_unique('questions_used.jsonl'), WORK / 'questions_used.jsonl')
archives = sorted(INPUT.rglob('checkpoint_130m.zip'))
assert len(archives) <= 1, 'Оставьте один Dataset с checkpoint_130m.zip'
if archives:
    shutil.copy2(archives[0], WORK / 'checkpoint_130m.zip')
else:
    # Kaggle может автоматически распаковать ZIP при загрузке Dataset.
    checkpoint = find_unique('model.safetensors').parent
    names = ['config.json', 'generation_config.json', 'model.safetensors',
             'tokenizer.json', 'training_meta.json', 'inference_config.json']
    assert all((checkpoint / name).is_file() for name in names), 'Нужны все шесть файлов исходного checkpoint'
    with zipfile.ZipFile(WORK / 'checkpoint_130m.zip', 'w', compression=zipfile.ZIP_STORED) as archive:
        for name in names:
            archive.write(checkpoint / name, 'best/' + name)
print('Входные файлы готовы в', WORK)


In [ ]:
from pathlib import Path
import os
BASE = Path('/content') if Path('/content').exists() else Path.cwd()
os.chdir(BASE)
QUESTIONS_FILE = 'questions_used.jsonl'
for name in ['checkpoint_130m.zip', QUESTIONS_FILE]:
    assert Path(name).is_file(), f'Загрузите {name} в {BASE}'
INFERENCE_PROGRAM = '"""Standalone cold-load inference benchmark, strict NVIDIA T4 by default.\n\nDependency installation and obtaining input artifacts precede the timed run.\nThe measurement includes imports, checkpoint extraction/loading, tokenization,\nGPU generation, and writing/checking the 1919-line output. No datasets library.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\nimport zipfile\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--checkpoint\', default=\'checkpoint_130m.zip\')\n    parser.add_argument(\'--questions\', default=\'questions_used.jsonl\')\n    parser.add_argument(\'--output\', default=\'submission_130m.jsonl\')\n    parser.add_argument(\'--allow-other-gpu\', action=\'store_true\')\n    parser.add_argument(\'--batch-size\', type=int)\n    parser.add_argument(\'--max-new-tokens\', type=int)\n    args = parser.parse_args()\n    begin = time.perf_counter()\n    import os\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    os.environ.setdefault(\'USE_TF\', \'0\')\n    os.environ.setdefault(\'USE_FLAX\', \'0\')\n    os.environ.setdefault(\'TOKENIZERS_PARALLELISM\', \'false\')\n    import math\n    import hashlib\n    import torch\n    from tokenizers import Tokenizer\n    from transformers import BartForConditionalGeneration\n    torch.set_num_threads(4)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime\')\n    gpu = torch.cuda.get_device_name()\n    if not args.allow_other_gpu and \'T4\' not in gpu:\n        raise RuntimeError(f\'This submission benchmark requires an actual T4; found {gpu}\')\n    archive_path = Path(args.checkpoint)\n    checkpoint_sha256 = None\n    if archive_path.is_file():\n        digest = hashlib.sha256()\n        with archive_path.open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                digest.update(chunk)\n        checkpoint_sha256 = digest.hexdigest()\n        target = Path(\'checkpoint_130m_extracted\')\n        with zipfile.ZipFile(archive_path) as archive:\n            allowed = {\'best/\' + n for n in [\'config.json\', \'generation_config.json\', \'model.safetensors\',\n                       \'tokenizer.json\', \'training_meta.json\', \'inference_config.json\']}\n            if set(archive.namelist()) != allowed:\n                raise ValueError(\'Unexpected checkpoint archive files\')\n            archive.extractall(target)\n        checkpoint = target / \'best\'\n    else:\n        checkpoint = archive_path\n    settings = json.loads((checkpoint / \'inference_config.json\').read_text())\n    if args.batch_size is not None:\n        settings[\'batch_size\'] = args.batch_size\n    if args.max_new_tokens is not None:\n        settings[\'max_new_tokens\'] = args.max_new_tokens\n    if settings[\'batch_size\'] < 1 or not 1 <= settings[\'max_new_tokens\'] <= settings[\'max_target\']:\n        raise ValueError(\'Invalid batch size or generation length\')\n    rows = [json.loads(line) for line in Path(args.questions).read_text().splitlines()]\n    questions = [row[\'question\'] for row in rows]\n    if len(questions) != 1919 or any(not isinstance(q, str) or not q.strip() for q in questions):\n        raise ValueError(\'Expected 1919 nonempty questions in original order\')\n    tokenizer = Tokenizer.from_file(str(checkpoint / \'tokenizer.json\'))\n    model = BartForConditionalGeneration.from_pretrained(\n        str(checkpoint), local_files_only=True, torch_dtype=torch.float16,\n        attn_implementation=\'sdpa\').cuda().eval()\n    parameters = sum(p.numel() for p in model.parameters())\n    if parameters != 130_183_936:\n        raise ValueError(\'Wrong model checkpoint\')\n    encoded = tokenizer.encode_batch(questions, add_special_tokens=False)\n    source_clipped = sum(len(x.ids) + 1 > settings[\'max_source\'] for x in encoded)\n    sequences = [x.ids[:settings[\'max_source\'] - 1] + [3] for x in encoded]\n    order = sorted(range(len(sequences)), key=lambda i: len(sequences[i]))\n    predictions = [None] * len(questions)\n    no_eos = 0\n    batch_size = settings[\'batch_size\']\n    with torch.inference_mode():\n        for start in range(0, len(order), batch_size):\n            indices = order[start:start + batch_size]\n            width = math.ceil(max(len(sequences[i]) for i in indices) / 8) * 8\n            ids = torch.zeros((len(indices), width), dtype=torch.long, device=\'cuda\')\n            for j, index in enumerate(indices):\n                ids[j, :len(sequences[index])] = torch.tensor(sequences[index], device=\'cuda\')\n            generated = model.generate(input_ids=ids, attention_mask=ids.ne(0).long(),\n                                       do_sample=False, num_beams=1, use_cache=True,\n                                       max_new_tokens=settings[\'max_new_tokens\'], forced_eos_token_id=None,\n                                       suppress_tokens=[0, 1, 2]).cpu().tolist()\n            for index, tokens in zip(indices, generated):\n                predictions[index] = tokenizer.decode(tokens, skip_special_tokens=True)\n                no_eos += 3 not in tokens[1:]\n    output = Path(args.output)\n    output.write_text(\'\'.join(json.dumps({\'code\': code}, ensure_ascii=False) + \'\\n\' for code in predictions))\n    checked = [json.loads(line) for line in output.read_text().splitlines()]\n    assert len(checked) == 1919 and all(set(row) == {\'code\'} and isinstance(row[\'code\'], str) for row in checked)\n    torch.cuda.synchronize()\n    seconds = time.perf_counter() - begin\n    result = {\'gpu\': gpu, \'parameters\': parameters, \'rows\': len(checked), \'cold_seconds\': seconds,\n              \'max_new_tokens\': settings[\'max_new_tokens\'], \'batch_size\': batch_size, \'no_eos\': no_eos,\n              \'source_clipped\': source_clipped, \'checkpoint_sha256\': checkpoint_sha256,\n              \'actual_t4\': \'T4\' in gpu, \'within_180_seconds\': seconds <= 180,\n              \'submission_sha256\': hashlib.sha256(output.read_bytes()).hexdigest(),\n              \'questions_sha256\': hashlib.sha256(Path(args.questions).read_bytes()).hexdigest()}\n    output.with_suffix(\'.timing.json\').write_text(json.dumps(result, indent=2))\n    print(json.dumps(result, indent=2))\n    if not args.allow_other_gpu and seconds > 180:\n        raise RuntimeError(\'T4 180-second requirement failed\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
BENCHMARK_PROGRAM = '"""Measure a fresh inference process on a real T4, without preloading Torch."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--checkpoint\', default=\'checkpoint_130m.zip\')\n    parser.add_argument(\'--questions\', default=\'questions_used.jsonl\')\n    parser.add_argument(\'--output\', default=\'submission_130m.jsonl\')\n    parser.add_argument(\'--batch-size\', type=int, default=128)\n    parser.add_argument(\'--max-new-tokens\', type=int, default=192)\n    parser.add_argument(\'--allow-other-gpu\', action=\'store_true\')\n    args = parser.parse_args()\n    command = [sys.executable, str(Path(__file__).with_name(\'inference_130m.py\')),\n               \'--checkpoint\', args.checkpoint, \'--questions\', args.questions,\n               \'--output\', args.output, \'--batch-size\', str(args.batch_size),\n               \'--max-new-tokens\', str(args.max_new_tokens)]\n    if args.allow_other_gpu:\n        command.append(\'--allow-other-gpu\')\n    timing = Path(args.output).with_suffix(\'.timing.json\')\n    timing.unlink(missing_ok=True)\n    started = time.perf_counter()\n    process = subprocess.run(command, timeout=190)\n    seconds = time.perf_counter() - started\n    if not timing.exists():\n        process.check_returncode()\n        raise RuntimeError(\'Inference did not write a timing result\')\n    result = json.loads(timing.read_text())\n    result.update(cold_subprocess_seconds=seconds,\n                  t4_limit_verified=result[\'actual_t4\'] and seconds <= 180,\n                  within_180_seconds=seconds <= 180)\n    timing.write_text(json.dumps(result, indent=2))\n    print(json.dumps(result, indent=2))\n    process.check_returncode()\n    if not args.allow_other_gpu and not result[\'t4_limit_verified\']:\n        raise RuntimeError(\'The real T4 run exceeded 180 seconds\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
Path("inference_130m.py").write_text(INFERENCE_PROGRAM)
Path("benchmark_130m.py").write_text(BENCHMARK_PROGRAM)


Таймер запускается до нового Python-процесса. Время включает импорты, проверку и распаковку архива, загрузку модели, токенизацию, генерацию 1919 программ и запись JSONL. Установка пакетов и загрузка входных файлов выполняются заранее. Проверка требует actual_t4=true и не больше 180 секунд.

In [ ]:
import subprocess, sys, json
BATCH_SIZE = 128
import zipfile
with zipfile.ZipFile("checkpoint_130m.zip") as archive:
    settings = json.loads(archive.read("best/inference_config.json"))
MAX_NEW_TOKENS = settings["max_new_tokens"]
print("Выбранный на валидации лимит:", MAX_NEW_TOKENS)
subprocess.run([sys.executable, "benchmark_130m.py", "--questions", QUESTIONS_FILE, "--batch-size", str(BATCH_SIZE), "--max-new-tokens", str(MAX_NEW_TOKENS)], check=True, timeout=210)
report = json.loads(Path("submission_130m.timing.json").read_text())
assert report["rows"] == 1919 and report["parameters"] == 130183936
assert report["actual_t4"] and report["t4_limit_verified"]
print("Все 1919 задач на T4:", report["cold_subprocess_seconds"], "секунд")


В конце должны быть rows=1919, actual_t4=true, t4_limit_verified=true и cold_subprocess_seconds <= 180. Скачайте оба файла по ссылкам ниже и сохраните выполненный ноутбук через Save Version → Save & Run All. При нехватке памяти измените BATCH_SIZE на 64 и повторите запуск.


In [ ]:
from IPython.display import FileLink, display
display(FileLink("submission_130m.jsonl"))
display(FileLink("submission_130m.timing.json"))
